## Notebook 10 – Combining Datasets

### Setup
For this notebook I only need pandas. The command below installs it. If it is already installed, pip just says "Requirement already satisfied", so it is safe to run again. I used %pip so that the library is installed in the same environment as this notebook.

In [1]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\rafiy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In real projects the data is almost never in one table. The customer details are in one table and the orders are in another, or the sales of every month are in a separate file. This notebook is about putting such tables together.

For the main examples I use two small tables of a shop. The customers table has 5 customers. The orders table has 6 orders. I made the tables in such a way that customer 6 (in the orders table) does not exist in the customers table, and customers 3 and 5 (Meena and Sana) have no orders. This helps to see the difference between the types of merge.

In [3]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "name": ["Asha", "Ravi", "Meena", "Kiran", "Sana"],
    "city": ["Hyderabad", "Chennai", "Pune", "Mumbai", "Hyderabad"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106],
    "customer_id": [1, 2, 2, 4, 6, 1],
    "product": ["Laptop", "Phone", "Tablet", "Phone", "Laptop", "Tablet"],
    "amount": [50, 20, 15, 20, 50, 15]
})

print(customers)
print()
print(orders)

   customer_id   name       city
0            1   Asha  Hyderabad
1            2   Ravi    Chennai
2            3  Meena       Pune
3            4  Kiran     Mumbai
4            5   Sana  Hyderabad

   order_id  customer_id product  amount
0       101            1  Laptop      50
1       102            2   Phone      20
2       103            2  Tablet      15
3       104            4   Phone      20
4       105            6  Laptop      50
5       106            1  Tablet      15


## Topic 1 – concat()

### Step 1 – Understand the Concept
pd.concat() joins tables by stacking them, either one below the other or side by side. It does not look at any key column. It only uses the position and the labels (index and column names).

Main options:
- axis=0 (default) : stack the tables one below the other (more rows).
- axis=1 : put the tables side by side (more columns). The rows are matched by their index labels, not by their position.
- ignore_index=True : gives new row numbers 0, 1, 2 ... to the result.
- keys=[...] : adds a label for each table, so the rows can be traced back to their source.
- join : "outer" (default) keeps all the columns (or rows), and "inner" keeps only the ones that all the tables have in common.

When the columns are different, the missing parts are filled with NaN.

### Step 2 – Demonstrate the Concept

#### Business Example
A shop has a separate sales table for each month. For the quarter report, the tables of January and February (and then March) are stacked one below the other to make a single table.

### Step 3 – Implement the Concept

In [4]:
jan = pd.DataFrame({"order_id": [1, 2, 3], "product": ["Pen", "Book", "Bag"], "units": [10, 5, 2]})
feb = pd.DataFrame({"order_id": [4, 5], "product": ["Pen", "Bag"], "units": [8, 3]})

# Stacking one below the other
stacked = pd.concat([jan, feb])
print(stacked)
print()
print("Shape:", stacked.shape)

# New row numbers
print()
print(pd.concat([jan, feb], ignore_index=True))

# Keeping track of the source with keys
tagged = pd.concat([jan, feb], keys=["jan", "feb"])
print()
print(tagged)
print()
print(tagged.loc["feb"])

   order_id product  units
0         1     Pen     10
1         2    Book      5
2         3     Bag      2
0         4     Pen      8
1         5     Bag      3

Shape: (5, 3)

   order_id product  units
0         1     Pen     10
1         2    Book      5
2         3     Bag      2
3         4     Pen      8
4         5     Bag      3

       order_id product  units
jan 0         1     Pen     10
    1         2    Book      5
    2         3     Bag      2
feb 0         4     Pen      8
    1         5     Bag      3

   order_id product  units
0         4     Pen      8
1         5     Bag      3


In [5]:
# Tables with different columns
mar = pd.DataFrame({"order_id": [6, 7], "product": ["Book", "Pen"],
                    "units": [4, 6], "discount": [5, 0]})

print(pd.concat([jan, mar], ignore_index=True))

# Keeping only the common columns
print()
print(pd.concat([jan, mar], ignore_index=True, join="inner"))

   order_id product  units  discount
0         1     Pen     10       NaN
1         2    Book      5       NaN
2         3     Bag      2       NaN
3         6    Book      4       5.0
4         7     Pen      6       0.0

   order_id product  units
0         1     Pen     10
1         2    Book      5
2         3     Bag      2
3         6    Book      4
4         7     Pen      6


In [6]:
# Side by side (axis=1): rows are matched by the index labels
personal = pd.DataFrame({"name": ["Asha", "Ravi", "Meena"]}, index=[1, 2, 3])
pay = pd.DataFrame({"salary": [40000, 65000, 58000]}, index=[2, 3, 4])

print(pd.concat([personal, pay], axis=1))

print()
print(pd.concat([personal, pay], axis=1, join="inner"))

    name   salary
1   Asha      NaN
2   Ravi  40000.0
3  Meena  65000.0
4    NaN  58000.0

    name  salary
2   Ravi   40000
3  Meena   65000


#### What does this program do?
In the first cell, jan has 3 rows and feb has 2 rows. concat([jan, feb]) stacks them into one table of shape (5, 3), but the row numbers are 0, 1, 2, 0, 1, because each table brought its own numbers. With ignore_index=True the numbers become 0 to 4. With keys=["jan", "feb"] the rows get an extra outer label, so tagged.loc["feb"] gives back only the 2 rows of February (orders 4 and 5).

In the second cell, the March table has an extra column, discount. When it is stacked with January, the result has 5 rows and 4 columns, and the discount is NaN for the 3 January rows (they did not have this column) and 5 and 0 for March. With join="inner" only the 3 common columns (order_id, product and units) are kept, so the shape is (5, 3).

In the third cell, the two tables are put side by side with axis=1. The index of personal is 1, 2, 3 and the index of pay is 2, 3, 4, so the rows are matched by these labels. The result has the index 1 to 4. Row 1 has a name but no salary (NaN) and row 4 has a salary but no name (NaN). With join="inner" only the rows 2 and 3 remain, which are Ravi with 40000 and Meena with 65000.

#### AI/ML Example
A big dataset is often delivered in many parts, like one file per day or per city. concat() with ignore_index=True joins all the parts into one training table. axis=1 is used to add new feature columns to a dataset, for example putting the original features next to the new features that were calculated separately (the two tables must have the same index).

#### Advantages
- Very simple way to stack many tables at once.
- Handles tables with different columns.
- keys makes it possible to remember where each row came from.

#### Limitations
- It does not match rows by a key column, so it cannot add the customer details to the orders (merge() is for that).
- Without ignore_index=True the row numbers can repeat.
- With axis=1 the rows are matched by index, so a mismatch in the index gives NaN values.

#### Key Point
concat() stacks tables below each other (axis=0) or side by side (axis=1). It uses position and labels, not a key column.

## Topic 2 – merge()

### Step 1 – Understand the Concept
pd.merge() combines two tables by matching the values of a key column, like a JOIN in SQL. Every row of one table is joined with the rows of the other table that have the same key value.

Main options:
- on="col" : the key column that has the same name in both tables.
- left_on and right_on : the key columns when the names are different.
- how : the type of the merge.
  - inner (default) : keeps only the keys that are in both tables.
  - left : keeps all the rows of the left table, and the right columns are NaN where there is no match.
  - right : keeps all the rows of the right table.
  - outer : keeps all the rows of both tables.
- suffixes : names added to the columns that have the same name in both tables (the default is _x and _y).
- indicator=True : adds a column _merge that tells if a row came from both tables, only the left or only the right.
- validate : checks the type of the relation (like "many_to_one") and gives an error if it is not true.

If a key value appears many times, every combination of the matching rows is created, so the number of rows can grow.

### Step 2 – Demonstrate the Concept

#### Business Example
The orders table has only the customer id. To see the name and the city of the customer for each order, the orders table is merged with the customers table using customer_id as the key.

### Step 3 – Implement the Concept

In [7]:
# Orders with customer details (inner merge by default)
merged = pd.merge(orders, customers, on="customer_id")
print(merged)
print()
print("Shape:", merged.shape)

   order_id  customer_id product  amount   name       city
0       101            1  Laptop      50   Asha  Hyderabad
1       102            2   Phone      20   Ravi    Chennai
2       103            2  Tablet      15   Ravi    Chennai
3       104            4   Phone      20  Kiran     Mumbai
4       106            1  Tablet      15   Asha  Hyderabad

Shape: (5, 6)


In [8]:
# The four types of merge: look at the number of rows
for how in ["inner", "left", "right", "outer"]:
    result = pd.merge(orders, customers, on="customer_id", how=how)
    print(how, result.shape)

# left merge: all the orders, customer 6 has no details
print()
print(pd.merge(orders, customers, on="customer_id", how="left"))

# right merge: all the customers, Meena and Sana have no orders
print()
print(pd.merge(orders, customers, on="customer_id", how="right"))

inner (5, 6)
left (6, 6)
right (7, 6)
outer (8, 6)

   order_id  customer_id product  amount   name       city
0       101            1  Laptop      50   Asha  Hyderabad
1       102            2   Phone      20   Ravi    Chennai
2       103            2  Tablet      15   Ravi    Chennai
3       104            4   Phone      20  Kiran     Mumbai
4       105            6  Laptop      50    NaN        NaN
5       106            1  Tablet      15   Asha  Hyderabad

   order_id  customer_id product  amount   name       city
0     101.0            1  Laptop    50.0   Asha  Hyderabad
1     106.0            1  Tablet    15.0   Asha  Hyderabad
2     102.0            2   Phone    20.0   Ravi    Chennai
3     103.0            2  Tablet    15.0   Ravi    Chennai
4       NaN            3     NaN     NaN  Meena       Pune
5     104.0            4   Phone    20.0  Kiran     Mumbai
6       NaN            5     NaN     NaN   Sana  Hyderabad


In [9]:
# Finding out which rows matched: indicator=True
check = pd.merge(orders, customers, on="customer_id", how="outer", indicator=True)
print(check["_merge"].value_counts())

# Key columns with different names
customers_alt = customers.rename(columns={"customer_id": "cust_id"})
alt = pd.merge(orders, customers_alt, left_on="customer_id", right_on="cust_id")
print()
print(alt.columns.tolist())

_merge
both          5
right_only    2
left_only     1
Name: count, dtype: int64

['order_id', 'customer_id', 'product', 'amount', 'cust_id', 'name', 'city']


In [10]:
# Columns with the same name in both tables: suffixes
shipping = pd.DataFrame({
    "customer_id": [1, 2, 4],
    "city": ["Hyderabad", "Bengaluru", "Mumbai"]
})

print(pd.merge(customers, shipping, on="customer_id").columns.tolist())

print()
print(pd.merge(customers, shipping, on="customer_id", suffixes=("_home", "_ship")))

['customer_id', 'name', 'city_x', 'city_y']

   customer_id   name  city_home  city_ship
0            1   Asha  Hyderabad  Hyderabad
1            2   Ravi    Chennai  Bengaluru
2            4  Kiran     Mumbai     Mumbai


In [11]:
# Repeated keys make the number of rows grow
dup_customers = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)
print("Customers table rows:", len(dup_customers), "(Asha is there twice)")
print("Rows after merge    :", len(pd.merge(orders, dup_customers, on="customer_id")))

# validate gives an error when the relation is not what we expect
try:
    pd.merge(orders, dup_customers, on="customer_id", validate="many_to_one")
except pd.errors.MergeError as error:
    print()
    print("MergeError:", error)

print()
print("Correct data passes the check:", len(pd.merge(orders, customers, on="customer_id", validate="many_to_one")), "rows")

Customers table rows: 6 (Asha is there twice)
Rows after merge    : 7

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  customer_id
           1 ...

Correct data passes the check: 5 rows


#### What does this program do?
In the first cell, the orders are merged with the customers on customer_id. The default type is inner, so only the matching keys remain. Order 105 belongs to customer 6, who is not in the customers table, so that order is dropped. The customers Meena and Sana have no orders, so they are not in the result either. The result has 5 rows and 6 columns, and each order now has the name and the city of its customer (101 Asha, 102 Ravi, 103 Ravi, 104 Kiran and 106 Asha).

In the second cell, the same merge is done with the four types, and the shapes are (5, 6) for inner, (6, 6) for left, (7, 6) for right and (8, 6) for outer. The left merge keeps all 6 orders, and order 105 has NaN for the name and the city. The right merge keeps all 5 customers, so Meena and Sana appear with NaN for the order columns, and Asha and Ravi appear twice because they have two orders each (7 rows in total). The outer merge has all of them, which is 8 rows.

In the third cell, indicator=True adds the column _merge. The counts show both 5, right_only 2 (Meena and Sana) and left_only 1 (order 105). This is a quick way to check which rows did not find a match. Then the key columns with different names are merged with left_on and right_on, and the result has both customer_id and cust_id as columns.

In the fourth cell, both the customers table and the shipping table have a city column, so by default the result gets the names city_x and city_y. With suffixes=("_home", "_ship") the names are city_home and city_ship, and the 3 customers who are in both tables (1, 2 and 4) are shown with the home city and the shipping city side by side. Ravi's home city is Chennai and his shipping city is Bengaluru.

In the last cell, Asha is added twice to the customers table. Now each of her 2 orders is matched with 2 rows, so the merge gives 7 rows instead of 5, which shows how repeated keys increase the row count. With validate="many_to_one", pandas gives a MergeError because the keys of the right table are not unique. With the correct customers table, the same check passes and gives 5 rows.

#### AI/ML Example
Machine learning data is usually spread over many tables, such as a transactions table and a users table. merge() adds the user features to every transaction before training. Two things must be checked after every merge: the number of rows (a left merge should keep the same number of rows as the left table) and the NaN values from unmatched keys. indicator=True and validate are used for these checks.

#### Advantages
- Matches the rows by key, so related data from different tables is put together correctly.
- The four types of merge give control over which rows to keep.
- indicator and validate help to find mistakes.

#### Limitations
- Duplicate keys can make the number of rows grow without any warning, unless validate is used.
- Wrong or dirty keys (different spelling, extra spaces, different data types) cause missing matches.
- Merging two very big tables needs a lot of memory.

#### Key Point
merge() joins two tables on a key column. Choose how (inner, left, right or outer) carefully, and always check the number of rows and the unmatched rows afterwards.

## Topic 3 – join()

### Step 1 – Understand the Concept
join() is a DataFrame method that combines tables using the index. It is a shorter way to do a merge when the key is already the index of the other table.

- df1.join(df2) matches the index of df1 with the index of df2. The default type is how="left".
- df1.join(df2, on="col") matches the column "col" of df1 with the index of df2.
- how can be "left", "right", "inner" or "outer", like in merge().
- If both tables have a column with the same name, lsuffix and rsuffix have to be given, otherwise there is an error.

join() and merge() give the same kind of result. The main differences are that join() works on the index by default, and its default type is left, while merge() works on columns by default and its default type is inner.

### Step 2 – Demonstrate the Concept

#### Business Example
The shop has a loyalty points table where the customer id is the index. To add the points to the customers table, both tables are joined by their index, and the customers who have no points get NaN.

### Step 3 – Implement the Concept

In [12]:
customers_idx = customers.set_index("customer_id")
loyalty = pd.DataFrame({"points": [120, 80, 45]}, index=[1, 2, 4])

# Left join by index (default): all the customers are kept
print(customers_idx.join(loyalty))

# Inner join: only the customers who are in both tables
print()
print(customers_idx.join(loyalty, how="inner"))

              name       city  points
customer_id                          
1             Asha  Hyderabad   120.0
2             Ravi    Chennai    80.0
3            Meena       Pune     NaN
4            Kiran     Mumbai    45.0
5             Sana  Hyderabad     NaN

              name       city  points
customer_id                          
1             Asha  Hyderabad     120
2             Ravi    Chennai      80
4            Kiran     Mumbai      45


In [13]:
# Using a column of the left table with the index of the right table
orders_with_names = orders.join(customers_idx, on="customer_id")
print(orders_with_names)
print()
print("Shape:", orders_with_names.shape)

   order_id  customer_id product  amount   name       city
0       101            1  Laptop      50   Asha  Hyderabad
1       102            2   Phone      20   Ravi    Chennai
2       103            2  Tablet      15   Ravi    Chennai
3       104            4   Phone      20  Kiran     Mumbai
4       105            6  Laptop      50    NaN        NaN
5       106            1  Tablet      15   Asha  Hyderabad

Shape: (6, 6)


In [14]:
# Columns with the same name need suffixes
shipping_idx = shipping.set_index("customer_id")

try:
    customers_idx.join(shipping_idx)
except ValueError:
    print("ValueError: the column city is in both tables, so a suffix is needed")

print()
print(customers_idx.join(shipping_idx, lsuffix="_home", rsuffix="_ship"))

ValueError: the column city is in both tables, so a suffix is needed

              name  city_home  city_ship
customer_id                             
1             Asha  Hyderabad  Hyderabad
2             Ravi    Chennai  Bengaluru
3            Meena       Pune        NaN
4            Kiran     Mumbai     Mumbai
5             Sana  Hyderabad        NaN


#### What does this program do?
In the first cell, customers_idx has the customer_id as its index, and the loyalty table has the ids 1, 2 and 4 as its index. The default left join keeps all 5 customers, and the points column has 120 for Asha, 80 for Ravi and 45 for Kiran, while Meena and Sana get NaN, because they are not in the loyalty table (the column shows decimals because of the NaN). With how="inner" only the 3 customers who have points remain.

In the second cell, orders.join(customers_idx, on="customer_id") matches the customer_id column of the orders with the index of the customers. The result has all 6 orders, with the name and the city added (shape (6, 6)), and order 105 has NaN for both because customer 6 does not exist. This gives the same table as the left merge of the previous topic.

In the third cell, the customers table and the shipping table both have a city column, so a plain join gives a ValueError (the message says that the columns overlap but no suffix was given). With lsuffix="_home" and rsuffix="_ship", the join works, and the result has the columns name, city_home and city_ship. Kiran's shipping city is Mumbai, and Meena and Sana have NaN as the shipping city because they are not in the shipping table.

#### AI/ML Example
When the features of the same samples are stored in different tables that share the same index (for example, the sample id), join() is a quick way to put them together into one table. It is also used to add the result of a group calculation (which has the group name as its index) back to the original table.

#### Advantages
- Short and clean when the key is already the index.
- Default left join keeps all the rows of the main table.
- Can use a column of the left table with the index of the right one (on=).

#### Limitations
- Works on the index, so the key column has to be set as the index first, otherwise merge() is simpler.
- Columns with the same name give an error unless suffixes are given.
- A beginner can confuse the defaults, because join() uses left and merge() uses inner.

#### Key Point
join() combines tables using the index, and its default type is left. Use on= to match a column with the index of the other table.

## Topic 4 – append() (deprecation and alternatives)

### Step 1 – Understand the Concept
DataFrame.append() was an older method to add the rows of one table (or a single row) at the end of another table, like df1.append(df2).

It is deprecated and removed:
- It was deprecated in pandas 1.4, and it gave a FutureWarning.
- It was removed completely in pandas 2.0. In the new versions of pandas, calling it gives an AttributeError, because the method does not exist any more.

Reasons for removing it:
- It did exactly what pd.concat() does, so there were two ways for the same job.
- It made a full copy of the data on every call. People used it inside loops to add one row at a time, which is very slow for big data, because the whole table is copied again and again.

Alternatives:
- pd.concat([df1, df2], ignore_index=True) to add the rows of one table to another.
- pd.concat([df, pd.DataFrame([row_dictionary])], ignore_index=True) to add a single row.
- df.loc[len(df)] = [...] to add one row at the end of a table (useful only for a few rows).
- To add many rows in a loop, collect the rows in a Python list, and create the DataFrame once at the end.

### Step 2 – Demonstrate the Concept

#### Business Example
A shop gets a new order and wants to add it to the January table. In older code this was written with append(). In the current pandas, the new order is turned into a small table and added with concat().

### Step 3 – Implement the Concept

In [15]:
# Does append exist in this version of pandas?
print("Has append method?", hasattr(jan, "append"))

try:
    result = jan.append(feb, ignore_index=True)
    print("Old pandas: append still works (with a FutureWarning)")
except AttributeError:
    print("append() does not exist in this pandas version, use concat() instead")

Has append method? False
append() does not exist in this pandas version, use concat() instead


In [16]:
# Alternative 1: concat for two tables
print(pd.concat([jan, feb], ignore_index=True))

# Alternative 2: concat for one new row
new_row = pd.DataFrame([{"order_id": 6, "product": "Book", "units": 4}])
print()
print(pd.concat([jan, new_row], ignore_index=True))

# Alternative 3: loc with the next row number
temp = jan.copy()
temp.loc[len(temp)] = [6, "Book", 4]
print()
print(temp)

   order_id product  units
0         1     Pen     10
1         2    Book      5
2         3     Bag      2
3         4     Pen      8
4         5     Bag      3

   order_id product  units
0         1     Pen     10
1         2    Book      5
2         3     Bag      2
3         6    Book      4

   order_id product  units
0         1     Pen     10
1         2    Book      5
2         3     Bag      2
3         6    Book      4


In [17]:
# Alternative 4 (best for many rows): collect the rows in a list and build the table once
rows = []
for i in range(1, 4):
    rows.append({"order_id": 10 + i, "product": "Pen", "units": i})

experiment_results = pd.DataFrame(rows)
print(experiment_results)

   order_id product  units
0        11     Pen      1
1        12     Pen      2
2        13     Pen      3


#### What does this program do?
The first cell checks if the table has an append method. In pandas 2 and the newer versions, hasattr() gives False and the try block fails with an AttributeError, so the message says that concat() has to be used. In an old pandas (below 2.0), the method exists and works with a warning, so the other message is printed.

In the second cell, the first alternative joins the January and February tables with concat() and gives 5 rows with the numbers 0 to 4. The second alternative turns a dictionary with the new order into a one row table and adds it, so the January table gets a 4th row with order 6 (Book, 4 units). The third alternative uses loc with the row number len(temp), which is 3, so the new row is added at the end as a 4th row. This is good for a few rows but not for many.

In the third cell, 3 dictionaries are collected in the list rows inside a loop, and the DataFrame is created only once at the end. The result has 3 rows with the order ids 11, 12 and 13 and the units 1, 2 and 3.

#### AI/ML Example
While running many experiments, for example testing different model settings, the result of each run (settings and score) is a row. The right way is to collect these rows in a list inside the loop and create the DataFrame once at the end, as in the last cell. Calling concat() or the old append() inside the loop copies the table every time and becomes very slow when there are many runs.

#### Advantages
- The alternatives (concat and building from a list) are faster and clearer than append().
- concat() can join many tables in a single call.

#### Limitations
- Old code that uses append() breaks with an AttributeError in pandas 2 and above, so it has to be changed.
- Adding rows one by one, with any method, is slow for big data, so the rows should be collected first.

#### Key Point
append() was removed in pandas 2.0. Use pd.concat() instead, and for many rows collect them in a list first and create the DataFrame once.

## Topic 5 – combine_first()

### Step 1 – Understand the Concept
combine_first() combines two tables by filling the missing values of the first table with the values from the second table. The tables are matched by the index and the column names.

- Where the first table has a value, that value is kept, even if the second table also has one. So the first table has the priority.
- Where the first table has NaN (or the row or column does not exist in it), the value from the second table is used.
- The result has the union of the rows and the columns of both tables, so rows that are only in the second table are also added.

It works on a Series as well. It is different from fillna(other_table), which fills only the missing values of the cells that already exist in the first table and does not add new rows.

### Step 2 – Demonstrate the Concept

#### Business Example
A company has a main customer record (CRM) with some missing phone numbers and cities, and also an older backup record. The main record is trusted more, but where it has gaps they should be filled from the backup, and the customers who are only in the backup should be added.

### Step 3 – Implement the Concept

In [18]:
crm = pd.DataFrame({
    "name": ["Asha", "Ravi", "Meena"],
    "phone": ["98111", None, "98333"],
    "city": ["Hyderabad", "Chennai", None]
}, index=[1, 2, 3])

backup = pd.DataFrame({
    "name": ["Ravi", "Meena", "Kiran"],
    "phone": ["98222", "98334", "98444"],
    "city": ["Chennai", "Pune", "Mumbai"]
}, index=[2, 3, 4])

print(crm)
print()
print(backup)

combined = crm.combine_first(backup)
print()
print(combined)
print()
print("Shape:", combined.shape)

    name  phone       city
1   Asha  98111  Hyderabad
2   Ravi    NaN    Chennai
3  Meena  98333        NaN

    name  phone     city
2   Ravi  98222  Chennai
3  Meena  98334     Pune
4  Kiran  98444   Mumbai

    name  phone       city
1   Asha  98111  Hyderabad
2   Ravi  98222    Chennai
3  Meena  98333       Pune
4  Kiran  98444     Mumbai

Shape: (4, 3)


In [19]:
# fillna with a table fills only the cells that already exist
filled = crm.fillna(backup)
print(filled)
print()
print("Shape with fillna        :", filled.shape)
print("Shape with combine_first :", combined.shape)

# The same idea for a Series
s1 = pd.Series([1, None, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"])
print()
print(s1.combine_first(s2))

    name  phone       city
1   Asha  98111  Hyderabad
2   Ravi  98222    Chennai
3  Meena  98333       Pune

Shape with fillna        : (3, 3)
Shape with combine_first : (4, 3)

a     1.0
b    20.0
c     3.0
d    40.0
dtype: float64


#### What does this program do?
In the first cell, crm has 3 customers (index 1, 2 and 3) with two gaps: Ravi's phone and Meena's city. The backup has the customers 2, 3 and 4. combine_first() keeps the values of crm wherever they exist and fills the gaps from the backup. The result has 4 rows (shape (4, 3)):
- Customer 1 (only in crm): Asha, 98111, Hyderabad.
- Customer 2: Ravi, the phone is filled from the backup (98222), and the city Chennai is from crm.
- Customer 3: Meena keeps her own phone 98333 (not the backup phone 98334), and the missing city is filled from the backup (Pune).
- Customer 4 (only in the backup): Kiran, 98444, Mumbai, who is added as a new row.

In the second cell, crm.fillna(backup) fills the same gaps (Ravi's phone and Meena's city), but the result has only 3 rows, because fillna() does not add customer 4, so the shape is (3, 3) while combine_first() gives (4, 3). At the end the Series s1 has a missing value at b and no value at d. combine_first() fills b with 20 from s2 and adds d with 40, and the result is 1.0, 20.0, 3.0 and 40.0 (decimals because of the NaN).

#### AI/ML Example
When the data comes from two sources, a main (trusted) source and a secondary one, combine_first() fills the gaps of the main source with the secondary values and adds the rows that only the second source has. It is also used to complete the missing feature values of a dataset from another dataset that covers the same samples, while keeping the original values untouched.

#### Advantages
- The priority of the first table is clear and simple.
- Fills the gaps and adds the new rows in one step.
- Works with both DataFrames and Series.

#### Limitations
- Matches by index and column names only, so the index must be set properly before.
- It cannot choose by a rule (for example "use the newer value"), the first table always wins when it has a value.
- The data type can change, for example whole numbers become decimals when NaN is present.

#### Key Point
combine_first(other) keeps the values of the first table, fills its gaps from the other table, and also adds the rows and columns that only the other table has.

## Interview Questions

#### 1. What is the difference between concat() and merge()?

concat() stacks tables below each other (axis=0) or side by side (axis=1) using the position and the index labels, and it does not need a key. merge() joins two tables by matching the values of a key column, like a SQL JOIN, so it is used to add related information (like customer details to orders) and it can create more or fewer rows depending on the matches.

#### 2. What are the types of merge? Explain with an example.

inner keeps only the keys that are in both tables, left keeps all the rows of the left table, right keeps all the rows of the right table, and outer keeps all the rows of both tables. When there is no match the other table's columns are NaN. For example, merging orders with customers: inner gives only the orders whose customer exists, left keeps all the orders (even those with an unknown customer), and outer also adds the customers who have no orders.

#### 3. What is the difference between join() and merge()?

Both combine tables by matching keys. join() matches on the index by default and its default type is left. merge() matches on columns by default and its default type is inner. join() is shorter when the key is already the index, and merge() is more flexible because it has options like left_on, right_on, indicator and validate.

#### 4. Why was append() removed and what should be used instead?

DataFrame.append() was deprecated in pandas 1.4 and removed in pandas 2.0, because it did the same job as concat() and it copied the full table on every call, which was very slow when used in a loop. Use pd.concat([df1, df2], ignore_index=True), and when many rows are added in a loop, collect them in a list and create the DataFrame once at the end.

#### 5. What is the difference between combine_first() and fillna()?

combine_first(other) keeps the values of the first table, fills its gaps from the other table and also adds the rows and columns that exist only in the other table (union of both). fillna(other) only fills the missing values in the cells that already exist in the first table, so the shape does not change and no new rows are added.